In [1]:
import pandas as pd

df = pd.read_csv("/Users/aliabonanno/public_policy_project/OpenEI/usurdb.csv.gz",
                 compression="gzip", nrows=5, low_memory=False)

print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(f"  {col}")

Shape: (5, 737)

Columns:
  label
  eiaid
  name
  is_default
  startdate
  enddate
  latest_update
  utility
  sector
  description
  servicetype
  source
  sourceparent
  dgrules
  peakkwcapacitymin
  peakkwcapacitymax
  peakkwcapacityhistory
  peakkwhusagemin
  peakkwhusagemax
  peakkwhusagehistory
  voltagecategory
  phasewiring
  voltageminimum
  voltagemaximum
  fixedchargefirstmeter
  fixedchargeeaaddl
  fixedchargeunits
  mincharge
  minchargeunits
  fixedattrs
  basicinformationcomments
  flatdemandunit
  flatdemandstructure/period0/tier0max
  flatdemandstructure/period0/tier0rate
  flatdemandstructure/period0/tier0adj
  flatdemandstructure/period0/tier0unit
  flatdemandstructure/period0/tier1max
  flatdemandstructure/period0/tier1rate
  flatdemandstructure/period0/tier1adj
  flatdemandstructure/period0/tier1unit
  flatdemandstructure/period0/tier2max
  flatdemandstructure/period0/tier2rate
  flatdemandstructure/period0/tier2adj
  flatdemandstructure/period0/tier2unit
  flatde

In [3]:
import pandas as pd
import numpy as np

# Load only the columns we need
cols = ["eiaid", "sector", "is_default", "startdate", "enddate",
        "fixedchargefirstmeter", "fixedchargeunits"]

df = pd.read_csv("/Users/aliabonanno/public_policy_project/OpenEI/usurdb.csv.gz",
                 compression="gzip", usecols=cols, low_memory=False)

print(f"Total rates: {len(df)}")
print(f"\nSector values:\n{df['sector'].value_counts().head(10)}")
print(f"\nFixed charge units:\n{df['fixedchargeunits'].value_counts().head(10)}")
print(f"\nIs default values:\n{df['is_default'].value_counts()}")
print(f"\nFixed charge sample (non-null):")
print(df[df["fixedchargefirstmeter"].notna()][
    ["eiaid","sector","is_default","fixedchargefirstmeter","fixedchargeunits"]
].head(10).to_string())

Total rates: 58275

Sector values:
sector
Commercial     27612
Residential    13620
Industrial     10688
Lighting        6344
Name: count, dtype: int64

Fixed charge units:
fixedchargeunits
$/month    52774
$/day       1187
Name: count, dtype: int64

Is default values:
is_default
False    12680
True       706
Name: count, dtype: int64

Fixed charge sample (non-null):
     eiaid       sector is_default  fixedchargefirstmeter fixedchargeunits
0  13799.0   Industrial        NaN                 100.62          $/month
1  18449.0  Residential        NaN                  25.00          $/month
2  16610.0   Commercial        NaN                  60.00          $/month
3  12439.0   Industrial        NaN                  59.00          $/month
4  15159.0  Residential        NaN                   8.00          $/month
5  18260.0   Commercial        NaN                  10.00          $/month
6  11522.0   Commercial        NaN                 145.79          $/month
7   5086.0     Lighting       

In [13]:
import os

tmax_dir = "/Users/aliabonanno/public_policy_project/prism/tmax"
files = sorted(os.listdir(tmax_dir))

sizes = []
for f in files:
    if f.endswith(".zip"):
        path = os.path.join(tmax_dir, f)
        size = os.path.getsize(path)
        sizes.append((f, size))

import pandas as pd
s = pd.Series([sz for _, sz in sizes])
print(f"Total zip files: {len(sizes)}")
print(f"\nSize distribution:")
print(s.describe())
print(f"\nFiles under 1000 bytes (likely corrupt): {(s < 1000).sum()}")
print(f"Files over 100KB (likely valid): {(s > 100000).sum()}")
print(f"\nSample of larger files:")
large = [(f, sz) for f, sz in sizes if sz > 100000][:5]
for f, sz in large:
    print(f"  {f}: {sz/1024:.0f} KB")

Total zip files: 3380

Size distribution:
count    3380.0
mean      430.0
std         0.0
min       430.0
25%       430.0
50%       430.0
75%       430.0
max       430.0
dtype: float64

Files under 1000 bytes (likely corrupt): 3380
Files over 100KB (likely valid): 0

Sample of larger files:


In [3]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

# One row per county (fixed charge is time-invariant)
counties = master.drop_duplicates(subset=["FIPS"])

print("Overall mean fixed charge:")
print(f"  ${counties['fixed_charge_county'].mean():.2f}/month")

print("\nMean fixed charge by ownership type:")
print(counties.groupby("dominant_ownership_type")["fixed_charge_county"]
      .agg(["mean", "median", "std", "count"]).round(2))

Overall mean fixed charge:
  $11.88/month

Mean fixed charge by ownership type:
                          mean  median   std  count
dominant_ownership_type                            
Cooperative              17.12   15.91  7.85    165
IOU                      11.32   10.78  3.78   2532
Municipal                16.71   17.66  5.33    117
